<a href="https://colab.research.google.com/github/Liza337/MSC-Emo-Sum/blob/main/Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =========================================================
# ABLATION STUDY: BANGLAT5 WITHOUT EMOTION LABELS
# Upload CSV from desktop in Google Colab
# =========================================================

# Install required packages
# !pip install transformers datasets evaluate bert-score -q
!pip install -q transformers datasets evaluate rouge_score bert-score

# ---------------------------------------------------------
# IMPORT LIBRARIES
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import torch

from google.colab import files
from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed
)

import evaluate
from bert_score import score as bertscore_score

# ---------------------------------------------------------
# REPRODUCIBILITY
# ---------------------------------------------------------
set_seed(42)

# ---------------------------------------------------------
# UPLOAD CSV FILE FROM DESKTOP
# ---------------------------------------------------------
uploaded = files.upload()

# Get uploaded file name automatically
file_name = list(uploaded.keys())[0]

# Read CSV
df = pd.read_csv(file_name)

print('Dataset loaded successfully!')
print(df.head())
print(df.columns)

# ---------------------------------------------------------
# OPTIONAL: RENAME COLUMNS IF NEEDED
# ---------------------------------------------------------
# Uncomment and edit if your column names are different
# df = df.rename(columns={
#     'Comment': 'comment',
#     'Summary': 'summary',
#     'Emotion': 'emotion'
# })

# ---------------------------------------------------------
# TRAIN-TEST SPLIT (80/20)
# ---------------------------------------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print('Train size:', len(train_df))
print('Test size :', len(test_df))

# ---------------------------------------------------------
# ABLATION SETTING: REMOVE EMOTION PREFIX
# ---------------------------------------------------------
train_df = train_df.copy()
test_df  = test_df.copy()

train_df['input_text'] = train_df['comment']
test_df['input_text']  = test_df['comment']

# ---------------------------------------------------------
# MODEL AND TOKENIZER
# ---------------------------------------------------------
MODEL_NAME = 'csebuetnlp/banglat5'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 64

# ---------------------------------------------------------
# TOKENIZATION FUNCTION
# ---------------------------------------------------------
def preprocess_function(batch):

    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding='max_length'
    )

    labels = tokenizer(
        text_target=batch['summary'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding='max_length'
    )

    labels_ids = labels['input_ids']
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels_ids
    ]

    model_inputs['labels'] = labels_ids
    return model_inputs

# ---------------------------------------------------------
# CREATE DATASETS
# ---------------------------------------------------------
train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

tokenized_train = train_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=train_df.columns.tolist()
)

tokenized_test = test_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=test_df.columns.tolist()
)

# ---------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

# ---------------------------------------------------------
# ROUGE METRICS
# ---------------------------------------------------------
rouge = evaluate.load('rouge')

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    labels_for_decode = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels_for_decode,
        skip_special_tokens=True
    )

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    result = {k: round(v * 100, 4) for k, v in result.items()}
    return result

# ---------------------------------------------------------
# TRAINING ARGUMENTS
# ---------------------------------------------------------
OUTPUT_DIR = './banglat5_without_emotion'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=200,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=False,
    report_to=[]
)

# ---------------------------------------------------------
# TRAINER
# ---------------------------------------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ---------------------------------------------------------
# TRAIN MODEL
# ---------------------------------------------------------
trainer.train()

# ---------------------------------------------------------
# ROUGE EVALUATION
# ---------------------------------------------------------
print('\nRunning ROUGE evaluation on TEST set...')
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print('\n===== BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====')
print(f'ROUGE-1 (F1): {test_results["eval_rouge1"]:.4f}%')
print(f'ROUGE-2 (F1): {test_results["eval_rouge2"]:.4f}%')
print(f'ROUGE-L (F1): {test_results["eval_rougeL"]:.4f}%')

# ---------------------------------------------------------
# GENERATE TEST SUMMARIES
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

preds = []
refs = []

batch_size_gen = 4
test_inputs = test_df['input_text'].tolist()
test_refs   = test_df['summary'].tolist()

for i in range(0, len(test_inputs), batch_size_gen):

    batch_inputs = test_inputs[i:i+batch_size_gen]

    enc = tokenizer(
        batch_inputs,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=MAX_INPUT_LEN
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **enc,
            max_new_tokens=MAX_TARGET_LEN,
            num_beams=4
        )

    for out in outputs:
        pred = tokenizer.decode(
            out,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )
        preds.append(pred.strip())

    refs.extend(test_refs[i:i+batch_size_gen])

# ---------------------------------------------------------
# BERTScore
# ---------------------------------------------------------
print('\nComputing BERTScore...')
P, R, F1 = bertscore_score(preds, refs, lang='bn', verbose=True)

print('\n===== BANGLAT5 WITHOUT EMOTION: BERTSCORE =====')
print(f'Precision: {P.mean().item():.4f}')
print(f'Recall   : {R.mean().item():.4f}')
print(f'F1       : {F1.mean().item():.4f}')

# ---------------------------------------------------------
# SAVE PREDICTIONS
# ---------------------------------------------------------
out_df = pd.DataFrame({
    'input': test_inputs,
    'reference': refs,
    'prediction': preds
})

out_csv = 'banglat5_without_emotion_predictions.csv'
out_df.to_csv(out_csv, index=False, encoding='utf-8-sig')

print(f'\nSaved predictions to: {out_csv}')

# ---------------------------------------------------------
# SAVE MODEL
# ---------------------------------------------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('\nModel saved to:', OUTPUT_DIR)

# ---------------------------------------------------------
# DOWNLOAD PREDICTIONS FILE
# ---------------------------------------------------------
files.download(out_csv)

Saving ICCIT2026 - Sheet1.csv to ICCIT2026 - Sheet1 (2).csv
Dataset loaded successfully!
   id                                            comment emotion  \
0   1  লড়াই করার এত সাহস হলে ছাত্রদের মতো নিরস্ত্র হ...   angry   
1   2  হ্যামলেট বাহিনীর এই নিউজ সাহসিকতার সাথে প্রচার...   happy   
2   3                 সরকারের কাছে জিজ্ঞেস করেন এরা কারা   angry   
3   4  সরকারি অস্ত্র ছাএলীগ হাতে কিভাবে এটা আমার সেনা...   angry   
4   5            দ্রুত সন্ত্রাসীদের আইনের আওতায় আনা হোক   angry   

                                             summary  
0  অস্ত্রধারীদের বিরুদ্ধে ক্ষোভ প্রকাশ করে শাস্তি...  
1  হ্যামলেট বাহিনীর সাহসী প্রচার নিয়ে আনন্দ প্রক...  
2  সরকারের কাছে ক্ষোভ নিয়ে প্রশ্ন তোলা হয়েছে এদ...  
3  সরকারি অস্ত্র ছাত্রলীগের হাতে কিভাবে এল তা নিয...  
4  সন্ত্রাসীদের দ্রুত শাস্তি দেওয়ার জন্য ক্রোধে ...  
Index(['id', 'comment', 'emotion', 'summary'], dtype='object')
Train size: 3200
Test size : 800


Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

done in 3.15 seconds, 253.64 sentences/sec

===== BANGLAT5 WITHOUT EMOTION: BERTSCORE =====
Precision: 0.7782
Recall   : 0.7695
F1       : 0.7735

Saved predictions to: banglat5_without_emotion_predictions.csv


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: ./banglat5_without_emotion


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from evaluate import load
rouge = load('rouge')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

preds = []
refs = []

for text, ref in zip(test_df['input_text'], test_df['summary']):

    enc = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=64,
            num_beams=4
        )

    pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    preds.append(pred)
    refs.append(ref)

# Check first few predictions
for i in range(5):
    print('\nPRED:', preds[i])
    print('REF :', refs[i])

# Compute ROUGE
result = rouge.compute(predictions=preds, references=refs)

print('\n===== BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====')
print(f'ROUGE-1 (F1): {result["rouge1"]*100:.4f}%')
print(f'ROUGE-2 (F1): {result["rouge2"]*100:.4f}%')
print(f'ROUGE-L (F1): {result["rougeL"]*100:.4f}%')

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs


PRED: নির্যাতন ও রাগ প্রকাশ
REF : রাগ বা ক্ষোভের সময় এমন হওয়া স্বাভাবিক বলে স্বীকার ও রাগ প্রকাশ।

PRED: মুসলিম পরিবারের সন্তান থাকার অধিকার নিয়ে বিস্ময় প্রকাশ।
REF : দেশের সকলের বসবাসের অধিকার নিয়ে বিস্ময় প্রকাশ

PRED: ইসলামের কথা না শোনার কারণে রাগ প্রকাশ।
REF : ধর্মের কাহিনী না শোনার কারণে ঘৃণা প্রকাশ।

PRED: সত্য কথা বলার জন্য রাগ প্রকাশ।
REF : ভাইসাবের এককভাবে সত্য কথা বলার প্রশংসা

PRED: প্রতিমা ভাঙার বিচার না হওয়ায় দুঃখ প্রকাশ।
REF : খুনের বিচার না পাওয়া দেশে মূর্তি ভাঙার বিচারও গুরুত্বপূর্ণ, বিস্ময় প্রকাশ

===== BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====
ROUGE-1 (F1): 0.0000%
ROUGE-2 (F1): 0.0000%
ROUGE-L (F1): 0.0000%


In [7]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False
)

r1, r2, rl = [], [], []

for pred, ref in zip(preds, refs):

    # Bangla tokenization by whitespace
    pred_tok = ' '.join(pred.split())
    ref_tok  = ' '.join(ref.split())

    scores = scorer.score(ref_tok, pred_tok)

    r1.append(scores['rouge1'].fmeasure)
    r2.append(scores['rouge2'].fmeasure)
    rl.append(scores['rougeL'].fmeasure)

print('\n===== BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====')
print(f'ROUGE-1 (F1): {np.mean(r1)*100:.4f}%')
print(f'ROUGE-2 (F1): {np.mean(r2)*100:.4f}%')
print(f'ROUGE-L (F1): {np.mean(rl)*100:.4f}%')


===== BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====
ROUGE-1 (F1): 0.0000%
ROUGE-2 (F1): 0.0000%
ROUGE-L (F1): 0.0000%


In [9]:
import pandas as pd
import evaluate

# Load saved predictions
out_df = pd.read_csv('banglat5_without_emotion_predictions.csv')

# Handle missing/nan values safely
preds = out_df['prediction'].fillna("").astype(str).tolist()
refs = out_df['reference'].fillna("").astype(str).tolist()

# Load rouge
rouge = evaluate.load('rouge')

# Calculate ROUGE without the English stemmer
rouge_results = rouge.compute(
    predictions=preds,
    references=refs,
    use_stemmer=False,
    tokenizer=lambda x: x.split()
)

print('\n===== CORRECTED BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====')
print(f'ROUGE-1 (F1): {rouge_results["rouge1"] * 100:.4f}%')
print(f'ROUGE-2 (F1): {rouge_results["rouge2"] * 100:.4f}%')
print(f'ROUGE-L (F1): {rouge_results["rougeL"] * 100:.4f}%')


===== CORRECTED BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====
ROUGE-1 (F1): 25.8101%
ROUGE-2 (F1): 9.0948%
ROUGE-L (F1): 25.3080%


In [10]:
!pip install -q sentence-transformers

In [11]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

# 1. Load your saved predictions CSV
csv_file = 'banglat5_without_emotion_predictions.csv'
df = pd.read_csv(csv_file)

# 2. Extract inputs, references, and predictions (handling any NaN values)
preds = df['prediction'].fillna("").astype(str).tolist()
refs = df['reference'].fillna("").astype(str).tolist()

print(f"Loaded {len(preds)} predictions from {csv_file}")

# 3. Load the LaBSE Model
print("Loading LaBSE model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
labse_model = SentenceTransformer('sentence-transformers/LaBSE', device=device)

# 4. Generate Embeddings
print("Encoding generated predictions and reference summaries...")
pred_embeddings = labse_model.encode(preds, convert_to_tensor=True, show_progress_bar=True)
ref_embeddings = labse_model.encode(refs, convert_to_tensor=True, show_progress_bar=True)

# 5. Calculate Pairwise Cosine Similarities (row-by-row matching)
# torch.diag extracts the similarity between pred[i] and ref[i]
cosine_scores = torch.nn.functional.cosine_similarity(pred_embeddings, ref_embeddings)
mean_labse_score = cosine_scores.mean().item()

# 6. Display Results
print('\n===== BANGLAT5 WITHOUT EMOTION: LABSE RESULTS =====')
print(f'LaBSE Mean Cosine Similarity: {mean_labse_score:.4f}')

# Optional: Add individual LaBSE scores to your dataframe and resave
df['labse_score'] = cosine_scores.cpu().numpy()
df.to_csv('banglat5_without_emotion_predictions_with_labse.csv', index=False, encoding='utf-8-sig')
print("Updated predictions with LaBSE scores saved to 'banglat5_without_emotion_predictions_with_labse.csv'")

Loaded 800 predictions from banglat5_without_emotion_predictions.csv
Loading LaBSE model...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Encoding generated predictions and reference summaries...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]


===== BANGLAT5 WITHOUT EMOTION: LABSE RESULTS =====
LaBSE Mean Cosine Similarity: 0.5760
Updated predictions with LaBSE scores saved to 'banglat5_without_emotion_predictions_with_labse.csv'
